# Importing Modules

In [1]:
!git clone https://github.com/Radhika-Amar-Desai/BSF_implementation.git

Cloning into 'BSF_implementation'...
remote: Enumerating objects: 224, done.
remote: Total 224 (delta 0), reused 0 (delta 0), pack-reused 224 (from 1)
Receiving objects: 100% (224/224), 371.61 MiB | 18.36 MiB/s, done.
Resolving deltas: 100% (106/106), done.
Updating files: 100% (41/41), done.


In [2]:
!git clone https://github.com/facebookresearch/dinov3

Cloning into 'dinov3'...
remote: Enumerating objects: 660, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 660 (delta 8), reused 2 (delta 2), pack-reused 625 (from 3)
Receiving objects: 100% (660/660), 13.10 MiB | 19.09 MiB/s, done.
Resolving deltas: 100% (215/215), done.


In [3]:
import sys
from pathlib import Path

# Add the project root (parent of notebooks) to Python's search path
sys.path.append(str(Path().resolve().parent))

import os
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# Configuration

In [4]:
# ACTIVATION_FOLDER = "data/rabit_activations_dinov3"
INPUT_DIM = 768
NUM_BLOCKS = 256
BLOCK_SIZE = 3
TOP_K = 8
BATCH_SIZE = 2048
LR = 3e-3
EPOCHS = 300
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Generating Activations

In [5]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 36.9 MB/s eta 0:00:00


In [6]:
!pip install einops

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import sys
import pathlib
import numpy as np
import torch
from torchvision.transforms import v2
import einops

POS_MEAN_PATH = '/content/BSF_implementation/pos_mean.npy'
# (n_patches, d)
POS_MEAN = np.load(POS_MEAN_PATH).astype(np.float32)

# Local repo settings
DINO_REPO = "/content/dinov3"
WEIGHTS = "/content/drive/MyDrive/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth"

# 1. FIX: Prepend the repository path so that hubconf.py can resolve internal modules
if DINO_REPO not in sys.path:
    sys.path.insert(0, DINO_REPO)

def load_rabbit_images(npz_path):
    """Load the rabbit images as a (N, H, W, 3) uint8 array."""
    return np.load(npz_path)['arr_0']

@torch.no_grad()
def dino_activations(
    images,
    *,
    device=None,
    batch_size=64,
):
    """
    Parameters
    ----------
    images : ndarray
        (N,H,W,3) uint8

    Returns
    -------
    ndarray
        (N,196,768) float32 patch embeddings
    """
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    if not pathlib.Path(WEIGHTS).exists():
        raise FileNotFoundError(
            f"Model weights not found at {WEIGHTS}. Please ensure the weights file "
            "'dinov3_vitb16_pretrain_lvd1689m.pth' is downloaded and placed in your Google Drive."
        )

    # 2. FIX: Initialize the empty backbone via Hub, then manually inject the state dict.
    # DINOv3's hubconf models do not directly accept a 'weights' string path argument.
    model = torch.hub.load(
        DINO_REPO,
        "dinov3_vitb16",
        source="local",
        pretrained=False  # Tells hubconf not to fetch from remote Meta servers
    )

    # Load and map the local pth checkpoint weights safely
    state_dict = torch.load(WEIGHTS, map_location="cpu")

    # Unwrap 'model' or 'teacher' key if saved from a training checkpoint wrapper
    if "model" in state_dict:
        state_dict = state_dict["model"]
    elif "teacher" in state_dict:
        state_dict = state_dict["teacher"]

    model.load_state_dict(state_dict, strict=True)
    model = model.to(device).eval()

    transform = v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        ),
    ])

    outputs = []

    for start in range(0, len(images), batch_size):
        batch = torch.stack(
            [transform(im) for im in images[start:start + batch_size]]
        ).to(device)

        feats = model.forward_features(batch)

        # Extract features (B, 196, 768) for ViT-B/16 @ 224x224
        patch_tokens = feats["x_norm_patchtokens"]
        outputs.append(patch_tokens.cpu())

    return torch.cat(outputs, dim=0).numpy()

def patch_grid(n_patches):
    """Side length of the square patch grid (14 for DINOv3 ViT-B/16 @ 224)."""
    g = int(round(n_patches ** 0.5))
    assert g * g == n_patches, f'{n_patches} patches is not a square grid'
    return g

# Rabbit images -> DINOv3 patch activations (fp32). Needs a GPU; ~1 min.
images = load_rabbit_images('/content/BSF_implementation/data/rabbit.npz')
acts = dino_activations(images)

# centre by the ImageNet per-position mean, flatten, scale so ||x|| ~ sqrt(d)
acts = acts - POS_MEAN
x = einops.rearrange(acts, 'n p d -> (n p) d')
x = x / np.sqrt((x ** 2).sum(1).mean()) * np.sqrt(x.shape[1])
grid = patch_grid(acts.shape[1])
print('activations:', x.shape, '  patch grid:', grid)

activations: (58800, 768)   patch grid: 14


# Dataset

In [10]:
class PatchDataset(Dataset):

    def __init__(self, activations):

        # Directly use the provided activations instead of loading from a folder
        self.samples = activations

        print("PatchDataset initialized with shape:", self.samples.shape)

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        x = self.samples[idx]

        return torch.from_numpy(x).float()

# DataLoader

In [11]:
# Use the generated activations 'x' directly for the PatchDataset
dataset = PatchDataset(x)

train_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)

PatchDataset initialized with shape: (58800, 768)


# MDL Formula

In [12]:
import math

import torch
import torch.nn as nn


class MDL(nn.Module):
    """
    Minimum Description Length (Eq. 5)

    Returns

        total_bits
        support_bits
        code_bits
        residual_bits
        dictionary_bits
    """

    def __init__(
        self,
        num_blocks,
        block_size,
        dataset_tokens,
        residual_sigma=1.0,
        code_sigma=1.0,
    ):
        super().__init__()

        self.G = num_blocks
        self.b = block_size
        self.N = dataset_tokens

        self.residual_sigma = residual_sigma
        self.code_sigma = code_sigma

    ###########################################################
    # log2(n choose k)
    ###########################################################

    @staticmethod
    def log2_binomial(n, k):

        lg = (
            math.lgamma(n + 1)
            - math.lgamma(k + 1)
            - math.lgamma(n - k + 1)
        )

        return lg / math.log(2)

    ###########################################################
    # Support cost
    ###########################################################

    def support_bits(self, active_mask):

        # active blocks for each sample

        k = active_mask.sum(dim=1)

        bits = []

        for kk in k.tolist():
            bits.append(
                self.log2_binomial(
                    self.G,
                    int(kk),
                )
            )

        return torch.tensor(
            bits,
            device=active_mask.device,
        )

    ###########################################################
    # Code cost
    ###########################################################

    def code_bits(self, z):

        # Assume Gaussian entropy

        bits = (
            z.pow(2).sum(dim=1)
            /
            (2 * self.code_sigma ** 2 * math.log(2))
        )

        return bits

    ###########################################################
    # Residual cost
    ###########################################################

    def residual_bits(
        self,
        x,
        reconstruction,
    ):

        residual = x - reconstruction

        bits = (
            residual.pow(2).sum(dim=1)
            /
            (2 * self.residual_sigma ** 2 * math.log(2))
        )

        return bits

    ###########################################################
    # Dictionary cost
    ###########################################################

    def dictionary_bits(self, decoder):

        weight = decoder.weight

        bits = (
            weight.numel() * 32
        ) / self.N

        return torch.tensor(
            bits,
            device=weight.device,
        )

    ###########################################################

    def forward(
        self,
        x,
        output,
        decoder,
    ):

        support = self.support_bits(
            output["active_mask"]
        )

        code = self.code_bits(
            output["z"]
        )

        residual = self.residual_bits(
            x,
            output["reconstruction"],
        )

        dictionary = self.dictionary_bits(
            decoder
        )

        total = (
            support
            + code
            + residual
            + dictionary
        )

        return {
            "total": total.mean(),
            "support": support.mean(),
            "code": code.mean(),
            "residual": residual.mean(),
            "dictionary": dictionary,
        }

# MDL For Grassmann BSF

In [33]:
import os
os.chdir("/content/BSF_implementation")

from improved_grassmann_bsf import build_model

In [34]:
model = build_model(
    input_dim=INPUT_DIM,
    num_blocks=NUM_BLOCKS,
    block_size=BLOCK_SIZE,
    top_k=TOP_K,
)


In [35]:
BSF_WEIGHTS_PATH = '/content/improved_grassmann_bsf_dinov3.pth'

if not pathlib.Path(BSF_WEIGHTS_PATH).exists():
    raise FileNotFoundError(
        f"BSF model weights not found at {BSF_WEIGHTS_PATH}. Please ensure the weights file is correctly placed."
    )

state_dict = torch.load(BSF_WEIGHTS_PATH, map_location="cpu")
model.load_state_dict(state_dict)
model.to(DEVICE).eval()

print(f"Successfully loaded BSF model weights from {BSF_WEIGHTS_PATH}")

Successfully loaded BSF model weights from /content/improved_grassmann_bsf_dinov3.pth


In [36]:
import math
import torch

model.eval()

####################################################
# MDL helper
####################################################

def log2_binomial(n, k):
    return (
        math.lgamma(n + 1)
        - math.lgamma(k + 1)
        - math.lgamma(n - k + 1)
    ) / math.log(2)


####################################################
# Parameters
####################################################

dataset_tokens = len(train_loader.dataset)

dictionary_bits = (
    model.D_raw.numel() * 32
) / dataset_tokens

support_bits_total = 0.0
code_bits_total = 0.0
residual_bits_total = 0.0

rss = 0.0
tss = 0.0

num_samples = 0

####################################################
# Evaluation
####################################################

with torch.no_grad():

    for x in train_loader:

        x = x.to(DEVICE)

        output = model(x)

        reconstruction = output["reconstruction"]

        ############################################
        # Support term
        ############################################

        active = output["active_mask"].sum(dim=1)

        for k in active.tolist():
            support_bits_total += log2_binomial(
                model.num_blocks,
                int(k),
            )

        ############################################
        # Code term
        ############################################

        z = output["z"]

        code_bits_total += (
            z.pow(2).sum().item()
            /
            (2 * math.log(2))
        )

        ############################################
        # Residual term
        ############################################

        residual = x - reconstruction

        residual_bits_total += (
            residual.pow(2).sum().item()
            /
            (2 * math.log(2))
        )

        ############################################
        # R²
        ############################################

        rss += residual.pow(2).sum().item()

        tss += (
            (x - x.mean(dim=0, keepdim=True))
            .pow(2)
            .sum()
            .item()
        )

        num_samples += x.size(0)

####################################################
# Average MDL
####################################################

avg_support = support_bits_total / num_samples
avg_code = code_bits_total / num_samples
avg_residual = residual_bits_total / num_samples

avg_mdl = (
    avg_support
    + avg_code
    + avg_residual
    + dictionary_bits
)

####################################################
# Average R²
####################################################

r2 = 1 - rss / tss

print(f"Average Support Bits    : {avg_support:.3f}")
print(f"Average Code Bits       : {avg_code:.3f}")
print(f"Average Residual Bits   : {avg_residual:.3f}")
print(f"Dictionary Bits         : {dictionary_bits:.3f}")
print("-" * 45)
print(f"Average MDL             : {avg_mdl:.3f} bits")
print(f"Reconstruction R²       : {r2:.4f}")

Average Support Bits    : 48.541
Average Code Bits       : 386.078
Average Residual Bits   : 90.652
Dictionary Bits         : 320.993
---------------------------------------------
Average MDL             : 846.265 bits
Reconstruction R²       : 0.7999


# MDL for Top-k SAE

In [37]:
from top_k_SAE import build_model

In [38]:
model = build_model(
    input_dim=INPUT_DIM,
    num_blocks=NUM_BLOCKS * BLOCK_SIZE,
    block_size=1,
    top_k=TOP_K * BLOCK_SIZE,
)

In [39]:
BSF_WEIGHTS_PATH = '/content/BSF_implementation/model_checkpoints/top_k_sae_model.pth'

if not pathlib.Path(BSF_WEIGHTS_PATH).exists():
    raise FileNotFoundError(
        f"BSF model weights not found at {BSF_WEIGHTS_PATH}. Please ensure the weights file is correctly placed."
    )

state_dict = torch.load(BSF_WEIGHTS_PATH, map_location="cpu")
model.load_state_dict(state_dict)
model.to(DEVICE).eval()

print(f"Successfully loaded BSF model weights from {BSF_WEIGHTS_PATH}")

Successfully loaded BSF model weights from /content/BSF_implementation/model_checkpoints/top_k_sae_model.pth


In [41]:
import math
import torch

model.eval()

####################################################
# MDL helper
####################################################

def log2_binomial(n, k):
    return (
        math.lgamma(n + 1)
        - math.lgamma(k + 1)
        - math.lgamma(n - k + 1)
    ) / math.log(2)


####################################################
# Parameters
####################################################

dataset_tokens = len(train_loader.dataset)

dictionary_bits = (
    model.decoder.weight.numel() * 32
) / dataset_tokens

support_bits_total = 0.0
code_bits_total = 0.0
residual_bits_total = 0.0

rss = 0.0
tss = 0.0

num_samples = 0

####################################################
# Evaluation
####################################################

with torch.no_grad():

    for x in train_loader:

        x = x.to(DEVICE)

        output = model(x)

        reconstruction = output["reconstruction"]

        ############################################
        # Support term
        ############################################

        active = output["active_mask"].sum(dim=1)

        for k in active.tolist():
            support_bits_total += log2_binomial(
                model.num_blocks,
                int(k),
            )

        ############################################
        # Code term
        ############################################

        z = output["z"]

        code_bits_total += (
            z.pow(2).sum().item()
            /
            (2 * math.log(2))
        )

        ############################################
        # Residual term
        ############################################

        residual = x - reconstruction

        residual_bits_total += (
            residual.pow(2).sum().item()
            /
            (2 * math.log(2))
        )

        ############################################
        # R²
        ############################################

        rss += residual.pow(2).sum().item()

        tss += (
            (x - x.mean(dim=0, keepdim=True))
            .pow(2)
            .sum()
            .item()
        )

        num_samples += x.size(0)

####################################################
# Average MDL
####################################################

avg_support = support_bits_total / num_samples
avg_code = code_bits_total / num_samples
avg_residual = residual_bits_total / num_samples

avg_mdl = (
    avg_support
    + avg_code
    + avg_residual
    + dictionary_bits
)

####################################################
# Average R²
####################################################

r2 = 1 - rss / tss

print(f"Average Support Bits    : {avg_support:.3f}")
print(f"Average Code Bits       : {avg_code:.3f}")
print(f"Average Residual Bits   : {avg_residual:.3f}")
print(f"Dictionary Bits         : {dictionary_bits:.3f}")
print("-" * 45)
print(f"Average MDL             : {avg_mdl:.3f} bits")
print(f"Reconstruction R²       : {r2:.4f}")

Average Support Bits    : 150.478
Average Code Bits       : 60.781
Average Residual Bits   : 74.391
Dictionary Bits         : 320.993
---------------------------------------------
Average MDL             : 606.643 bits
Reconstruction R²       : 0.8358


# MDL for Vanilla BSF

In [43]:
from vanilla_bsf import build_model

In [44]:
model = build_model(
    input_dim=INPUT_DIM,
    num_blocks=NUM_BLOCKS,
    block_size=BLOCK_SIZE,
    top_k=TOP_K,
)

In [45]:
BSF_WEIGHTS_PATH = '/content/BSF_implementation/model_checkpoints/improved_vanilla_bsf_dinov3.pth'

if not pathlib.Path(BSF_WEIGHTS_PATH).exists():
    raise FileNotFoundError(
        f"BSF model weights not found at {BSF_WEIGHTS_PATH}. Please ensure the weights file is correctly placed."
    )

state_dict = torch.load(BSF_WEIGHTS_PATH, map_location="cpu")
model.load_state_dict(state_dict)
model.to(DEVICE).eval()

print(f"Successfully loaded BSF model weights from {BSF_WEIGHTS_PATH}")

Successfully loaded BSF model weights from /content/BSF_implementation/model_checkpoints/improved_vanilla_bsf_dinov3.pth


In [50]:
import math
import torch

model.eval()

####################################################
# MDL helper
####################################################

def log2_binomial(n, k):
    return (
        math.lgamma(n + 1)
        - math.lgamma(k + 1)
        - math.lgamma(n - k + 1)
    ) / math.log(2)


####################################################
# Parameters
####################################################

dataset_tokens = len(train_loader.dataset)

dictionary_bits = (
    model.D.weight.numel() * 32
) / dataset_tokens

support_bits_total = 0.0
code_bits_total = 0.0
residual_bits_total = 0.0

rss = 0.0
tss = 0.0

num_samples = 0

####################################################
# Evaluation
####################################################

with torch.no_grad():

    for x in train_loader:

        x = x.to(DEVICE)

        output = model(x)

        reconstruction = output["reconstruction"]

        ############################################
        # Support term
        ############################################

        active = output["active_mask"].sum(dim=1)

        for k in active.tolist():
            support_bits_total += log2_binomial(
                NUM_BLOCKS * BLOCK_SIZE,
                int(k),
            )

        ############################################
        # Code term
        ############################################

        z = output["z"]

        code_bits_total += (
            z.pow(2).sum().item()
            /
            (2 * math.log(2))
        )

        ############################################
        # Residual term
        ############################################

        residual = x - reconstruction

        residual_bits_total += (
            residual.pow(2).sum().item()
            /
            (2 * math.log(2))
        )

        ############################################
        # R²
        ############################################

        rss += residual.pow(2).sum().item()

        tss += (
            (x - x.mean(dim=0, keepdim=True))
            .pow(2)
            .sum()
            .item()
        )

        num_samples += x.size(0)

####################################################
# Average MDL
####################################################

avg_support = support_bits_total / num_samples
avg_code = code_bits_total / num_samples
avg_residual = residual_bits_total / num_samples

avg_mdl = (
    avg_support
    + avg_code
    + avg_residual
    + dictionary_bits
)

####################################################
# Average R²
####################################################

r2 = 1 - rss / tss

print(f"Average Support Bits    : {avg_support:.3f}")
print(f"Average Code Bits       : {avg_code:.3f}")
print(f"Average Residual Bits   : {avg_residual:.3f}")
print(f"Dictionary Bits         : {dictionary_bits:.3f}")
print("-" * 45)
print(f"Average MDL             : {avg_mdl:.3f} bits")
print(f"Reconstruction R²       : {r2:.4f}")

Average Support Bits    : 61.328
Average Code Bits       : 91.627
Average Residual Bits   : 80.285
Dictionary Bits         : 320.993
---------------------------------------------
Average MDL             : 554.232 bits
Reconstruction R²       : 0.8228
